# Graviton SDK: Fluid Dynamics & Weather Research and Forecasting (WRF)
This notebook demonstrates how to evaluate WRF 4.8.0 atmospheric modeling workloads, upload a NetCDF virtual world file using `gv.load_WRF()`, and execute multi-node MPI simulations on Google Cloud Batch.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CodeZeroLabs/graviton-examples/blob/master/graviton_sdk_wrf_example.ipynb)

In [ ]:
# 1. Install the Graviton SDK
!pip install graviton-sdk

In [ ]:
import graviton as gv
import tempfile
import os

# 2. Authenticate with Graviton
gv.login("a-little-more-art-than-science")

## 1. Defining the WRF Simulation Domain
WRF simulation payloads require grid resolution parameters (`dx`, `dy`), grid dimensions (`e_we`, `e_sn`), and simulation duration (`sim_time`).

In [ ]:
wrf_payload = {
    "t": "wrf",
    "dx": 10000,      # Grid spacing in X direction (10 km)
    "dy": 10000,      # Grid spacing in Y direction (10 km)
    "e_we": 100,      # Grid dimension West-East
    "e_sn": 100,      # Grid dimension South-North
    "sim_time": 86400 # Simulation duration in seconds (24 hours)
}

# Evaluate domain complexity and obtain signed upload URL for the world file
eval_data = gv.evaluate(wrf_payload)
print(f"Job ID: {eval_data.id}")
print(f"Estimated Credit Cost: {eval_data.cost:.2f} Credits")
print(f"Estimated Duration: {eval_data.estimated_time:.2f}s")
print(f"Signed Upload URL: {eval_data.upload_url[:50]}...")

## 2. Uploading the NetCDF Virtual World File
WRF atmospheric simulations run inside a virtual world defined in a NetCDF binary file (`world.nc`). The SDK provides `gv.load_WRF(job_id, world_file_path)` which securely streams the NetCDF file directly to Graviton storage via the evaluation's signed URL.

In [ ]:
# Create a sample NetCDF world file for demonstration
temp_world_file = os.path.join(tempfile.gettempdir(), "demo_world.nc")
with open(temp_world_file, "wb") as f:
    f.write(b"CDF\x02" + b"\x00" * 1024)

# Upload the world file to the job's signed URL
print(f"Uploading world file '{temp_world_file}' for Job '{eval_data.id}'...")
upload_status = gv.load_WRF(eval_data.id, temp_world_file)
print(f"Upload Status: {upload_status}")

## 3. Launching the Multi-Node MPI Supercomputing Job
Once the world file has been uploaded, initiate compute on Google Cloud Batch across MPI-interconnected C2 nodes.

In [ ]:
# Trigger MPI cluster compute
job = gv.compute(eval_data.id)
print(f"Job Status: {job.status()}")

# Wait for completion and view results
result = job.wait()
print("Simulation Completed!")
print(f"Output URL: {result.get('output_url')}")